# Entrenamiento Federado — Herbario (UNAL + Melbourne) en Colab GPU

Espacio de embedding compartido: un modelo YOLOv8n de **17 clases** (UNAL 0–5, Melbourne 6–16),
entrenado con FedAvg sincronizado y máscara de clase por institución.

**Antes de empezar, sube a tu Google Drive:**
1. La carpeta **`FederatedModel`** con los `.py` (`class_mask.py`, `client.py`, `server.py`, `Trainer.py`, `metrics.py`, `VisualizationTools.py`, `AnaliticTools.py`, `privacy.py`, `model_comparison.py`).
2. Los datos comprimidos: **`data_UN.zip`** y **`data_MELU.zip`** (cada uno con su `train/images` y `train/labels` dentro).

> Activa la GPU: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**.


## 1. GPU + instalar dependencias

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
!nvidia-smi -L
!pip -q install ultralytics

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configura tus rutas
Edita estas 4 rutas para que apunten a donde subiste las cosas en tu Drive.

In [ ]:
# === EDITA ESTAS RUTAS ===
CODE_DIR      = "/content/drive/MyDrive/herbario/FederatedModel"    # carpeta con los .py
DATA_UN_ZIP   = "/content/drive/MyDrive/herbario/data_UN.zip"       # zip de data_UN
DATA_MELU_ZIP = "/content/drive/MyDrive/herbario/data_MELU.zip"     # zip de data_MELU
OUTPUT_DIR    = "/content/drive/MyDrive/herbario/federated_output"  # modelos salen aquí (persiste)

import os
assert os.path.isdir(CODE_DIR), f'No existe CODE_DIR: {CODE_DIR}'
for z in [DATA_UN_ZIP, DATA_MELU_ZIP]:
    assert os.path.isfile(z), f'No existe el zip: {z}'
print('rutas OK')

## 4. Descomprimir datos a disco local de Colab (rápido)

In [ ]:
import zipfile, os
os.makedirs('/content/data', exist_ok=True)
for z in [DATA_UN_ZIP, DATA_MELU_ZIP]:
    print('Descomprimiendo', z, '...')
    with zipfile.ZipFile(z) as f:
        f.extractall('/content/data')
print('listo')

## 5. Localizar automáticamente images/labels de cada institución

In [ ]:
import glob, os
def find_images_dir(root, inst):
    cands = [d for d in glob.glob(root + '/**/images', recursive=True)
             if os.path.isdir(d) and os.path.isdir(d.replace('/images', '/labels'))]
    cands.sort(key=lambda d: (inst.lower() not in d.lower(), len(d)))
    return cands[0] if cands else None

UN_IMAGES   = find_images_dir('/content/data', 'UN')
MELU_IMAGES = find_images_dir('/content/data', 'MELU')
assert UN_IMAGES and MELU_IMAGES, f'No encontré images/labels. UN={UN_IMAGES} MELU={MELU_IMAGES}'
print('UNAL images :', UN_IMAGES, '->', len(glob.glob(UN_IMAGES + '/*')), 'archivos')
print('MELU images :', MELU_IMAGES, '->', len(glob.glob(MELU_IMAGES + '/*')), 'archivos')

## 6. Escribir los configs YAML (17 clases globales)

In [ ]:
import yaml
UN_NAMES = {0:'Mascara_Codigos', 1:'Mascara_ColorChecker', 2:'Mascara_Descripciones',
            3:'Mascara_Encabezados', 4:'Mascara_Escalas', 5:'Mascara_Sellos'}
MELU_NAMES = {0:'small database label', 1:'handwritten data', 2:'stamp', 3:'annotation label',
              4:'scale', 5:'swing tag', 6:'full database label', 7:'database label',
              8:'swatch', 9:'institutional label', 10:'number'}

yaml.safe_dump({'train': UN_IMAGES,   'val': UN_IMAGES,   'names': UN_NAMES},
               open('/content/config_un.yaml', 'w'), allow_unicode=True, sort_keys=False)
yaml.safe_dump({'train': MELU_IMAGES, 'val': MELU_IMAGES, 'names': MELU_NAMES},
               open('/content/config_melu.yaml', 'w'), allow_unicode=True, sort_keys=False)
print('configs escritos en /content/')

## 7. Preparar el entorno (sys.path + carpeta de runs)

In [ ]:
import sys, os
sys.path.insert(0, CODE_DIR)
os.environ['HERBARIO_RUNS_DIR'] = '/content/runs'   # runs temporales (no en Drive)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('CODE_DIR en sys.path; runs -> /content/runs; modelos -> ', OUTPUT_DIR)

## 8. Entrenar el modelo federado (GPU)
En GPU puedes usar `imgsz=640, batch=16` y muchas más rondas. Ajusta `rounds` según el tiempo
que tengas. Cada 5 rondas verás el `mAP50` global — debe **subir** ronda a ronda.

In [ ]:
from class_mask import SharedClassSpace
from client import SharedEmbeddingClient
from server import EnhancedFederatedServer, FedAvg
from Trainer import EnhancedFederatedTrainer
from metrics import MetricsTracker

space = SharedClassSpace({'client1': list(range(0, 6)), 'client2': list(range(6, 17))})
space.assign_names({'client1': UN_NAMES, 'client2': MELU_NAMES})

clients = [SharedEmbeddingClient('client1', '/content/config_un.yaml',   space),
           SharedEmbeddingClient('client2', '/content/config_melu.yaml', space)]

server = EnhancedFederatedServer(
    FedAvg(), num_classes=17, class_names=space.get_global_names(),
    model_save_path=OUTPUT_DIR,
    metrics_tracker=MetricsTracker(save_dir=OUTPUT_DIR),
)

trainer = EnhancedFederatedTrainer(
    server, clients,
    rounds=40, epochs_per_round=2, local_images=600,
    imgsz=640, batch=16, workers=2,      # GPU: ajustes completos
    local_lr=0.01, warmup_epochs=0.0, weighting='data',
    eval_every=5, eval_images=300,
)
trainer.train()
print('\nMODELO FINAL:', server.last_save_path)

## 9. Verificación rápida: ¿detecta AMBAS instituciones?

In [ ]:
from model_comparison import load_yolo_model
from collections import Counter
import glob

m = load_yolo_model(str(server.last_save_path))
def dets(imgs):
    c = Counter()
    for im in imgs:
        r = m.predict(im, conf=0.25, verbose=False)[0]
        if r.boxes is not None:
            for k in r.boxes.cls.int().tolist(): c[k] += 1
    return c

un = sorted(glob.glob(UN_IMAGES + '/*'))[:40]
ml = sorted(glob.glob(MELU_IMAGES + '/*'))[:40]
du, dm = dets(un), dets(ml)
print('En imgs UNAL  -> UNAL(0-5):', sum(v for k,v in du.items() if k<6),
      '| MELU(6-16):', sum(v for k,v in du.items() if k>=6))
print('En imgs MELU  -> UNAL(0-5):', sum(v for k,v in dm.items() if k<6),
      '| MELU(6-16):', sum(v for k,v in dm.items() if k>=6))
print('\nUn federado sano detecta 0-5 en imgs UNAL y 6-16 en imgs Melbourne.')

## 10. Listo
El modelo global (`global_model_round_*.pt`) queda en tu **`OUTPUT_DIR` en Drive** (state_dict, se
carga con `model_comparison.load_yolo_model`). Descárgalo desde Drive, o para el comparativo por clase
completo corre `model_comparison.py` con ese checkpoint como `--federated`.

**Consejos:**
- Si el `mAP50` sigue subiendo en la ronda 40, sube `rounds` (p.ej. 80) y vuelve a correr.
- Colab gratis corta sesiones inactivas: no cierres la pestaña durante el entrenamiento.
- Guardar en Drive (`OUTPUT_DIR`) hace que el modelo persista aunque se reinicie la sesión.


---
# 11. Informe comparativo — Federado vs modelos locales

Compara, **clase por clase**, el modelo federado contra el modelo local de cada institución
(precisión, recall, mAP@0.5, mAP@0.5:0.95 y mIoU) y genera los gráficos PNG/PDF para la tesis.

**Requisito:** sube tus dos modelos locales a Drive, por ejemplo en
`MyDrive/herbario/local_models/`:

```
MyDrive/herbario/local_models/
├── yolo_trained_unal/best.pt     (modelo local UNAL, 6 clases)
└── yolo_trained_melu/best.pt     (modelo local Melbourne, 11 clases)
```


In [ ]:
# ---- Rutas de los modelos a comparar ----
import os, glob, re

LOCAL_UNAL = "/content/drive/MyDrive/herbario/local_models/yolo_trained_unal/best.pt"
LOCAL_MELU = "/content/drive/MyDrive/herbario/local_models/yolo_trained_melu/best.pt"

# Modelo federado = checkpoint de la ronda más alta guardada en OUTPUT_DIR
ckpts = glob.glob(os.path.join(OUTPUT_DIR, 'global_model_round_*.pt'))
assert ckpts, f'No hay checkpoints federados en {OUTPUT_DIR}'
FEDERATED = max(ckpts, key=lambda p: int(re.search(r'round_(\d+)_', p).group(1)))

for p in [LOCAL_UNAL, LOCAL_MELU]:
    assert os.path.isfile(p), f'Falta el modelo local: {p}'

print('Federado :', os.path.basename(FEDERATED))
print('Local UNAL:', LOCAL_UNAL)
print('Local MELU:', LOCAL_MELU)

### Ejecutar la comparación
Evalúa 4 combinaciones (local/federado × UNAL/Melbourne) sobre los sets de validación.
Con `compute_iou=True` además calcula el IoU medio de localización (más lento; ponlo en
`False` si quieres solo P/R/mAP).

In [ ]:
from class_mask import SharedClassSpace
from model_comparison import FederatedComparison

# Reconstruimos el espacio de clases (por si reiniciaste el runtime)
space_cmp = SharedClassSpace({'client1': list(range(0, 6)), 'client2': list(range(6, 17))})
space_cmp.assign_names({'client1': UN_NAMES, 'client2': MELU_NAMES})

REPORT_DIR = os.path.join(OUTPUT_DIR, 'comparison_results')

cmp = FederatedComparison(
    shared_class_space=space_cmp,
    client_configs={'client1': '/content/config_un.yaml',
                    'client2': '/content/config_melu.yaml'},
    client_display_names={'client1': 'UNAL', 'client2': 'Melbourne'},
    save_dir=REPORT_DIR,
    imgsz=640, batch=16,
)

cmp.run(
    local_model_paths={'client1': LOCAL_UNAL, 'client2': LOCAL_MELU},
    federated_model_path=FEDERATED,
    compute_iou=True,      # False = más rápido (omite el IoU medio)
)
print('\nInforme generado en:', REPORT_DIR)

### Tabla resumen (promedio macro por institución)

In [ ]:
import csv, statistics as st
from collections import defaultdict

CSV = os.path.join(REPORT_DIR, 'comparison_metrics.csv')
rows = list(csv.DictReader(open(CSV)))
agg = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
for r in rows:
    agg[r['institucion']][r['modelo']][r['metrica']].append(float(r['valor']))

print('=' * 64)
print('PROMEDIO MACRO — Local vs Federado')
print('=' * 64)
for inst in agg:
    print(f'\n{inst}:')
    for m in ['precision', 'recall', 'mAP50', 'mAP50-95', 'mIoU']:
        lo = agg[inst]['local'].get(m, [])
        fe = agg[inst]['federated'].get(m, [])
        if not lo and not fe:
            continue
        lm = st.mean(lo) if lo else float('nan')
        fm = st.mean(fe) if fe else float('nan')
        print(f'  {m:9s}  local={lm:.3f}   federado={fm:.3f}   delta={fm-lm:+.3f}')

### Tabla por clase

In [ ]:
per = defaultdict(dict)
for r in rows:
    per[(r['institucion'], r['clase'], r['modelo'])][r['metrica']] = float(r['valor'])

for inst in agg:
    print('\n' + '=' * 78)
    print(f'{inst} — mAP@0.5 por clase')
    print('=' * 78)
    print(f"{'clase':30s} {'local':>8s} {'federado':>10s} {'delta':>9s}")
    clases = sorted({c for (i, c, k) in per if i == inst})
    for c in clases:
        lo = per.get((inst, c, 'local'), {}).get('mAP50')
        fe = per.get((inst, c, 'federated'), {}).get('mAP50')
        lo_s = f'{lo:.3f}' if lo is not None else '  -  '
        fe_s = f'{fe:.3f}' if fe is not None else '  -  '
        d_s  = f'{fe-lo:+.3f}' if (lo is not None and fe is not None) else '   -   '
        print(f'{c[:30]:30s} {lo_s:>8s} {fe_s:>10s} {d_s:>9s}')

### Gráficos (los mismos PNG/PDF quedan guardados en Drive para la tesis)

In [ ]:
from IPython.display import Image, display

orden = (sorted(glob.glob(REPORT_DIR + '/macro_summary.png')) +
         sorted(glob.glob(REPORT_DIR + '/*panel_all_metrics.png')) +
         sorted(glob.glob(REPORT_DIR + '/*_delta.png')))
for p in orden:
    print('\n' + os.path.basename(p))
    display(Image(filename=p))

print('\nArchivos generados:')
for p in sorted(glob.glob(REPORT_DIR + '/*')):
    print(' ', os.path.basename(p))

### Cómo leerlo

- **`macro_summary.png`** — vista general: local vs federado en cada institución.
- **`client1_panel_all_metrics.png`** (UNAL) y **`client2_panel_all_metrics.png`** (Melbourne) —
  panel con las 5 métricas por clase.
- **`*_delta.png`** — barras divergentes: verde = el federado mejora, rojo = empeora.
- **`comparison_metrics.csv`** — todos los valores crudos (para tablas en LaTeX).

Un federado sano debe dar métricas **razonables en ambas instituciones**. Si una institución
sale cerca de 0, esa mitad de la cabeza no se entrenó — revisa la curva de `mAP50` del
entrenamiento y aumenta `rounds`.
